# GEDI L4A aboveground-biomass footprints (NASA Earthdata)

Download GEDI L4A footprint granules from the ORNL DAAC — a **vector** product (per-shot aboveground biomass), so `OUTPUT_KIND='vector'` and the facade rejects `aggregate=`. The backend fetches whole HDF5 orbit granules (no server-side subsetting in the MVP), so a wide window pulls several hundred MB; keep the window small. Live query — needs the `[earthdata]` extra and EDL credentials; wrapped for nbval-lax safety offline.

In [ ]:
from pathlib import Path

from earthlens import EarthLens

OUT_DIR = Path('earthdata_output')
OUT_DIR.mkdir(exist_ok=True)

In [ ]:
paths = None
try:
    paths = EarthLens(
        data_source='earthdata',
        dataset='GEDI_L4A_AGB_Density_V2_1_2056', variables=['agbd'],
        start='2020-05-01',
        end='2020-05-02',
        aoi=[12.0, -3.0, 18.0, 3.0],
        path=str(OUT_DIR),
    ).download(progress_bar=False)
    print(len(paths), 'granule(s):', [Path(p).name for p in paths])
except Exception as exc:
    print(f'skipped live query: {type(exc).__name__}: {exc}')

In [ ]:
# Each granule is an HDF5 file of per-beam footprints (lat/lon + agbd). Read a
# sample from the first beam and scatter it. Wrapped so an h5py / structure
# surprise does not mask the download above.
if paths:
    try:
        import h5py
        import matplotlib.pyplot as plt

        with h5py.File(paths[0], 'r') as h5:
            beam = next(k for k in h5 if k.startswith('BEAM'))
            lat = h5[f'{beam}/lat_lowestmode'][:5000]
            lon = h5[f'{beam}/lon_lowestmode'][:5000]
            agbd = h5[f'{beam}/agbd'][:5000]
        fig, ax = plt.subplots(figsize=(8, 6))
        sc = ax.scatter(lon, lat, c=agbd, cmap='YlGn', s=6, vmin=0)
        fig.colorbar(sc, ax=ax, label='AGBD (Mg/ha)')
        ax.set_title(f'GEDI L4A footprints ({beam}) — aboveground biomass')
        ax.set_xlabel('Longitude')
        ax.set_ylabel('Latitude')
        plt.tight_layout()
        plt.show()
    except Exception as exc:
        print(f'read/plot step skipped: {type(exc).__name__}: {exc}')